# Spectral photon camera

A three-dimensional camera view of light transported through dispersive glass, a fluorescent coating, a Rayleigh-scattering medium, and a TPB-coated PMT. Every simulated photon contributes to the illumination maps; the camera renders surfaces and scattered light.

Choose **2.5 million photons** to begin. Orbit the camera, compare the scenes, then increase the photon budget to reduce noise. The rendered brightness is normalized by the emitted count. Exposure is a separate display control.

Use the **TriChroma (GPU, pimm-bench)** kernel to export the scenes. The browser GPU renders the interactive view.

In [ ]:
from pathlib import Path
import sys
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "chroma-lite").is_dir() and (p / "chroma-lar").is_dir()), None)
if root is None:
    raise RuntimeError("Start Jupyter inside the trichroma checkout")
for source in (root / "chroma-lite", root / "chroma-lar"):
    if str(source) not in sys.path:
        sys.path.insert(0, str(source))
from chroma_lar.photon_camera import export_camera_bundle, notebook_camera_html
output = root / "notebooks" / "photon_camera_bundle"
catalog = export_camera_bundle(output)
print("Ready:", ", ".join(scene["name"] for scene in catalog["scenes"]))

In [ ]:
from IPython.display import HTML, display
display(HTML(notebook_camera_html(output)))

The ceiling area light and main beam both contribute physically transported photons; their source weights keep the expected brightness fixed when the photon budget changes. The prism and fluorescent scenes include a weak, explicitly simulated Rayleigh haze so light can be seen from the side. Ultraviolet photons are invisible in the camera until wavelength shifting produces visible light. The maps retain 64 wavelength bands and polarization moments; the camera accounts for wavelength-dependent refraction and attenuation.

This is a steady-state photon-density estimate with finite spatial and spectral resolution. It includes the first diffuse reflection from the room walls. Display color uses the bundled CIE matching functions and an exposure setting; it is not an absolute photometric calibration. The [diagnostic notebook](optical_showcase.ipynb) retains individual-event spectra, delay distributions, and trajectory checks.

The PMT scene uses the repository R5912 profile and synthetic TPB/glass/photocathode tables. The cutaway affects camera visibility only; all forward photons see the complete PMT. Any selected UV false-color view is labeled and is not visible radiance.

### Download the browser bundle

The embedded renderer uses the same shaders and tables as the portable bundle. If your Jupyter service blocks executable HTML output, download this ZIP, extract it on your computer, and run `python -m http.server 8765` inside the extracted directory. Open `http://localhost:8765/camera.html` in a WebGPU-capable browser.

In [ ]:
import shutil
from IPython.display import FileLink
archive = shutil.make_archive(str(output), "zip", root_dir=output)
display(FileLink(str(Path(archive).relative_to(Path.cwd()))))